# CRAM plans on Stretch (Isaac Sim, real-robot mode)

Drive the Stretch robot in the Isaac Sim apartment through **CRAM** plans: you
state *what* to achieve -- "park the arm", "raise the torso", "navigate to this
pose" -- as designators, and CRAM binds them to concrete robot motions at run
time, picking the Stretch-specific motion mappings and streaming them to the
giskard control server (`stretch_apartment_giskard_server.py`).

**Kernel**: select **CRAM**.

## Start the simulation and giskard server

Two background processes: the Isaac Sim scene and the giskard control server.
`cram_vrb_lab.control.launcher` starts each and waits for its ready marker. Skip this cell if you
already started them; stale instances are killed first so they do not fight over
the topics. Pass `terminal=True` to open each in a `gnome-terminal` window
instead of the background.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parent  # this notebook lives in demos/
sys.path.insert(0, str(REPO))

from cram_vrb_lab.control.launcher import start_isaac_sim, start_giskard_server, stop

sim_proc = start_isaac_sim(camera="both")          # terminal=True to open a gnome-terminal window
giskard_proc = start_giskard_server()

starting isaac sim, logging to /tmp/apartment_sim.log
isaac sim ready after 18s
starting giskard server, logging to /tmp/giskard_server.log
giskard server ready after 14s


## Connect and build a CRAM `Context`

CRAM plans run against a `Context`: a world, the robot in it, a ROS node, and the
set of robot-specific motion mappings. We fetch the world the giskard server
built (robot + apartment) over its `fetch_world` service and keep it live with a
`WorldSynchronizer`, then wrap it in a `Context`.

`alternative_motion_mappings` is the list of Stretch-specific motions; CRAM picks
from it by robot type and execution type, so passing the full set is safe.
`evaluate_conditions=False` skips the action pre/post-condition checks (they need
collision queries that are noisy against the coarse apartment model).

In [ ]:
import threading

import nest_asyncio
import rclpy
from rclpy.executors import MultiThreadedExecutor

nest_asyncio.apply()  # CRAM's REAL execution calls GiskardWrapper.execute,
                      # which run_until_completes inside the already-running kernel loop.

from coraplex.datastructures.dataclasses import Context
from coraplex.alternative_motion_mappings.stretch_motion_mapping import (
    StretchMoveToolCenterPoint,
    StretchMoveSim,
    StretchMoveReal,
    StretchClose,
)
from semantic_digital_twin.robots.stretch import Stretch
from semantic_digital_twin.adapters.ros.world_fetcher import fetch_world_from_service
from semantic_digital_twin.adapters.ros.world_synchronizer import WorldSynchronizer

STRETCH_MOTION_MAPPINGS = [
    StretchMoveToolCenterPoint,
    StretchMoveSim,
    StretchMoveReal,
    StretchClose,
]

if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node('cram_demo_node')
executor = MultiThreadedExecutor()
executor.add_node(node)
threading.Thread(target=executor.spin, daemon=True, name='rclpy-executor').start()

# The world the giskard server published (Stretch + apartment). 300 s matches
# giskardpy's own client: the server may still be parsing the URDF on first start.
world = fetch_world_from_service(node=node, timeout_seconds=300)
WorldSynchronizer(_world=world, node=node)

robot = world.get_semantic_annotations_by_type(Stretch)
robot = robot[0] if robot else Stretch.from_world(world)

context = Context(
    world=world,
    robot=robot,
    ros_node=node,
    evaluate_conditions=False,
    alternative_motion_mappings=STRETCH_MOTION_MAPPINGS,
)
print('connected, robot:', type(robot).__name__)

## A small run helper

`with real_robot(...)` sets the execution type to REAL, so `plan.perform()`
builds each action's giskard motion and streams it to the running server (which
drives Isaac). `collision_avoidance=True` adds an `ExternalCollisionAvoidance`
goal against the apartment the world carries.

In [ ]:
from coraplex.execution_environment import real_robot
from coraplex.plans.factories import sequential, execute_single


def run_plan(plan, collision_avoidance=True):
    """Perform a CRAM plan on the real (sim) robot via giskard."""
    with real_robot(collision_avoidance=collision_avoidance):
        plan.perform()
    print('done')

## 1. Body: park the arm and raise the torso

`ParkArmsAction` and `MoveTorsoAction` are intent-level: CRAM looks up Stretch's
parked arm configuration and torso joint from the semantic model and turns each
into a `MoveJointsMotion` -- no joint names or target values in the plan.

In [ ]:
from coraplex.robot_plans.actions.core.robot_body import (
    ParkArmsAction,
    MoveTorsoAction,
)
from coraplex.datastructures.enums import Arms
from semantic_digital_twin.datastructures.definitions import TorsoState

run_plan(sequential([
    ParkArmsAction(Arms.LEFT),
    MoveTorsoAction(TorsoState.MID),
], context=context))

## 2. Gripper open / close

`SetGripperAction` maps the semantic `GripperState` to the finger joint targets
(0.109 open / 0.0 closed for Stretch).

In [ ]:
from coraplex.robot_plans.actions.core.robot_body import SetGripperAction
from semantic_digital_twin.datastructures.definitions import GripperState

run_plan(execute_single(SetGripperAction(Arms.LEFT, GripperState.OPEN), context=context))

In [ ]:
run_plan(execute_single(SetGripperAction(Arms.LEFT, GripperState.CLOSE), context=context))

## 3. Navigate the base to a pose (with apartment collision avoidance)

`NavigateAction` takes a target `Pose` in the world and CRAM resolves it to a
Stretch base motion (`StretchMoveReal` for REAL execution). With
`collision_avoidance=True` the whole-body QP keeps a margin from the apartment
walls/furniture that the shared world carries.

`world.root` is the `map` frame (identical to the Isaac world frame here), so the
target is in world coordinates. The robot spawns near `(-1.5, 0)`; pick a nearby
free spot and adjust if the robot refuses to move (it may be starting too close
to a collision body).

In [ ]:
from coraplex.robot_plans.actions.core.navigation import NavigateAction
from semantic_digital_twin.spatial_types.spatial_types import Pose
from semantic_digital_twin.spatial_types import Point3

target = Pose(Point3.from_iterable([-1.0, -0.8, 0.0]), reference_frame=world.root)
run_plan(execute_single(NavigateAction(target), context=context))

## Next step: object manipulation

`TransportAction(obj, target_pose, Arms.LEFT)` would drive the whole pick-and-
place from one line. It is left out here because the object has to exist in
**both** places to work: as a body in this digital twin (so CRAM can plan the
grasp) **and** as a rigid body in the Isaac scene (so the gripper has something to
close on). Spawning a box in `stretch_apartment_sim.py` and merging the matching body into the
world is the next step.

## Shutdown

Stop the server and the simulation (only if they were started from this notebook).

In [ ]:
# stop()  # stops the isaac sim + giskard server started above